# Diabetes Prediction Dataset - Exploratory Data Analysis (EDA)

**College Project | Machine Learning**

This notebook explores the diabetes prediction dataset before model training. It covers data overview, class imbalance, feature distributions, correlations, and key insights for the final ML pipeline.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_URL = (
    "https://raw.githubusercontent.com/gopiashokan/dataset/main/"
    "diabetes_prediction_dataset.csv"
)

df = pd.read_csv(DATA_URL)
print(f"Dataset shape: {df.shape}")
df.head()

## 1. Dataset Overview

In [ ]:
print("Column Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())
print("\nStatistical Summary:")
df.describe().T

## 2. Target Variable Analysis (Class Imbalance)

In [ ]:
target_counts = df["diabetes"].value_counts()
target_pct = df["diabetes"].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_pct.round(2),
})
summary.index = ["No Diabetes (0)", "Diabetes (1)"]
print(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="diabetes", palette="Set2", ax=axes[0])
axes[0].set_title("Diabetes Class Distribution")
axes[0].set_xlabel("Diabetes (0 = No, 1 = Yes)")
axes[0].set_ylabel("Count")

axes[1].pie(
    target_counts,
    labels=["No Diabetes", "Diabetes"],
    autopct="%1.1f%%",
    colors=["#66c2a5", "#fc8d62"],
    startangle=90,
)
axes[1].set_title("Diabetes Percentage Share")
plt.tight_layout()
plt.show()

**Insight:** The dataset is imbalanced (~91.5% non-diabetic, ~8.5% diabetic). Accuracy alone is not enough; use F1 score and ROC-AUC during model evaluation.

## 3. Numeric Feature Distributions

In [ ]:
numeric_cols = [
    "age",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level",
]

df[numeric_cols].hist(bins=30, figsize=(12, 8), color="#4C72B0", edgecolor="white")
plt.suptitle("Distribution of Numeric Features", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap

In [ ]:
corr_df = df.copy()
corr_df["gender_code"] = corr_df["gender"].astype("category").cat.codes
corr_df["smoking_code"] = corr_df["smoking_history"].astype("category").cat.codes

corr_cols = [
    "gender_code",
    "age",
    "hypertension",
    "heart_disease",
    "smoking_code",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level",
    "diabetes",
]

plt.figure(figsize=(10, 7))
sns.heatmap(
    corr_df[corr_cols].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5,
)
plt.title("Feature Correlation Heatmap")
plt.show()

**Insight:** HbA1c level and blood glucose level show the strongest correlation with diabetes.

## 5. Feature vs Diabetes (Boxplots)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_features = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]

for ax, feature in zip(axes.flatten(), plot_features):
    sns.boxplot(data=df, x="diabetes", y=feature, palette="Set2", ax=ax)
    ax.set_title(f"{feature} vs Diabetes")
    ax.set_xlabel("Diabetes (0 = No, 1 = Yes)")

plt.suptitle("Numeric Features by Diabetes Status", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Categorical Feature Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gender_diabetes = (
    df.groupby(["gender", "diabetes"]).size().unstack(fill_value=0)
)
gender_diabetes.plot(kind="bar", ax=axes[0], color=["#66c2a5", "#fc8d62"])
axes[0].set_title("Diabetes Count by Gender")
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Count")
axes[0].legend(["No Diabetes", "Diabetes"])

smoking_rate = df.groupby("smoking_history")["diabetes"].mean().sort_values(ascending=False)
smoking_rate.plot(kind="bar", color="#4C72B0", ax=axes[1])
axes[1].set_title("Diabetes Rate by Smoking History")
axes[1].set_xlabel("Smoking History")
axes[1].set_ylabel("Diabetes Rate")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 7. Hypertension & Heart Disease Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, feature, title in zip(
    axes,
    ["hypertension", "heart_disease"],
    ["Hypertension vs Diabetes Rate", "Heart Disease vs Diabetes Rate"],
):
    rate = df.groupby(feature)["diabetes"].mean()
    sns.barplot(x=rate.index, y=rate.values, palette="viridis", ax=ax)
    ax.set_title(title)
    ax.set_xlabel(feature.replace("_", " ").title())
    ax.set_ylabel("Diabetes Rate")
    ax.set_xticklabels(["No", "Yes"])

plt.tight_layout()
plt.show()

## 8. Key EDA Conclusions

1. **Dataset size:** 1,00,000 records with no missing values.
2. **Class imbalance:** Only ~8.5% patients have diabetes.
3. **Strong predictors:** HbA1c level and blood glucose level are most related to diabetes.
4. **Supporting factors:** BMI, age, hypertension, and heart disease also contribute.
5. **Next step:** Train models using stratified split, balanced weights, and compare F1/ROC-AUC.

Run `python train_model.py` to train the final model and `streamlit run app.py` for the web app.